# Authenticated Product Data Extraction for Procurement Agents

Extract structured product data from any commerce URL with per-field confidence scores using ShopGraph's authenticated extraction API. Confidence-aware routing separates verified data from fields that need human review.

In [ ]:
%pip install requests langchain langchain-openai

In [ ]:
import os
import json
import requests

SHOPGRAPH_API_KEY = os.environ.get("SHOPGRAPH_API_KEY", "your-api-key")
SHOPGRAPH_URL = "https://shopgraph.dev/api/enrich"

In [ ]:
def extract_product(url: str) -> dict:
    """Extract structured product data with confidence scores."""
    response = requests.post(
        SHOPGRAPH_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {SHOPGRAPH_API_KEY}",
        },
        json={"url": url},
    )

    # Error handling: return structured error, don't raise
    if response.status_code != 200:
        return {"error": True, "status_code": response.status_code, "message": f"Extraction failed for {url}"}

    data = response.json()
    if "product" not in data:
        return {"error": True, "status_code": response.status_code, "message": f"No product data returned for {url}"}

    return data


data = extract_product("https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8")

# Response structure:
# data["product"]["product_name"]  -> "DAYTON 1/2 HP Jet Pump, Model 5UXK1"
# data["product"]["price"]["amount"]  -> 284.00
# data["product"]["price"]["currency"]  -> "USD"
# data["product"]["availability"]  -> "in_stock"
# data["product"]["confidence"]["overall"]  -> 0.93
# data["product"]["confidence"]["per_field"]["price"]  -> 0.93
# data["product"]["_shopgraph"]["field_confidence"]["price"]  -> 0.93
# data["product"]["_shopgraph"]["field_freshness"]["price"]["decayed"]  -> False
# data["cached"]  -> False
# data["credit_mode"]  -> "standard"

print(f"Product: {data.get('product', {}).get('product_name', 'N/A')}")

## Confidence-Aware Routing

Per-field confidence scores let you programmatically route data into different handling paths:

- **Verified** (confidence >= threshold): Safe for automation
- **Needs review** (confidence >= 0.50 but < threshold): Flag for human check
- **Missing** (confidence < 0.50 or absent): Request manual entry

This is the pattern that separates a brittle pipeline from a reliable one.

In [ ]:
def extract_with_confidence_routing(
    url: str, confidence_threshold: float = 0.8
) -> dict:
    """
    Extract product data and route fields into verified/review/missing buckets
    based on per-field confidence scores.
    """
    data = extract_product(url)

    # If extraction returned an error, pass it through
    if "error" in data:
        return data

    product = data["product"]
    meta = product["_shopgraph"]

    verified = {}
    needs_review = {}
    missing = {}

    field_map = {
        "product_name": product.get("product_name"),
        "brand": product.get("brand"),
        "description": product.get("description"),
        "price": product.get("price", {}).get("amount") if product.get("price") else None,
        "currency": product.get("price", {}).get("currency") if product.get("price") else None,
        "availability": product.get("availability"),
        "categories": product.get("categories"),
        "primary_image_url": product.get("primary_image_url"),
        "material": product.get("material"),
    }

    for field_name, value in field_map.items():
        if value is None or value == [] or value == "unknown":
            missing[field_name] = {"value": value, "reason": "not_available"}
            continue

        # Look up confidence from _shopgraph.field_confidence
        confidence = meta["field_confidence"].get(field_name, 0)

        entry = {"value": value, "confidence": confidence}

        # Check freshness for real-time fields
        freshness = meta.get("field_freshness", {}).get(field_name)
        if freshness and freshness.get("decayed"):
            entry["decayed"] = True
            needs_review[field_name] = entry
        elif confidence >= confidence_threshold:
            verified[field_name] = entry
        else:
            needs_review[field_name] = entry

    return {
        "url": product["url"],
        "extraction_method": meta["extraction_method"],
        "data_source": meta["data_source"],
        "overall_confidence": product["confidence"]["overall"],
        "verified": verified,
        "needs_review": needs_review,
        "missing": missing,
    }

In [ ]:
result = extract_with_confidence_routing(
    "https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8",
    confidence_threshold=0.85,
)

if "error" in result:
    print(f"Error: {result['message']}")
else:
    print("Verified fields:")
    for field, info in result["verified"].items():
        print(f"  {field}: {info['value']} (confidence: {info['confidence']})")

    print("\nNeeds review:")
    for field, info in result["needs_review"].items():
        print(f"  {field}: {info['value']} (confidence: {info['confidence']})")

    print("\nMissing:")
    for field, info in result["missing"].items():
        print(f"  {field}: {info['reason']}")

## Using with a LangChain Agent

Wrap the confidence-routed extraction as a LangChain tool so an agent can extract and reason about product data quality.

## Two extraction modes: human-in-the-loop vs. autofill

Procurement agents make two kinds of calls:

1. **Research calls** where a human reviews the output. The agent wants every field, including low-confidence ones, so the human can judge what to trust. Use `extract_product`.

2. **Autofill calls** where the agent writes data directly into a PO, inventory system, or RFQ response with no human review. The agent wants *only* fields confident enough to act on — anything below threshold should be absent, not flagged.

ShopGraph supports the second mode via `strict_confidence_threshold`, a server-side parameter that scrubs low-confidence fields from the response before it leaves the API. The agent never sees fields below the threshold. This prevents "confident-but-wrong" writes: the agent cannot accidentally autofill a stale price it shouldn't have had in the first place.

Below we define a second tool, `extract_product_for_autofill`, and update the system prompt so the agent chooses deliberately between the two.

In [ ]:
@tool
def extract_product_for_autofill(url: str, threshold: float = 0.9) -> str:
    """Extract product data returning only fields above the given confidence
    threshold. Use this when writing extracted values directly into a purchase
    order, inventory system, or RFQ response without human review.

    Fields below threshold are scrubbed server-side and will not appear in
    the response. Missing fields mean 'not confident enough to act on.'

    Args:
        url: Product page URL.
        threshold: Minimum confidence (0.0-1.0). Default 0.9. Use 0.95 for
            contract pricing or regulated goods.
    """
    response = requests.post(
        SHOPGRAPH_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {SHOPGRAPH_API_KEY}",
        },
        json={"url": url, "strict_confidence_threshold": threshold},
        timeout=30,
    )
    response.raise_for_status()
    return json.dumps(response.json(), indent=2)


In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.prompts import ChatPromptTemplate


@tool
def extract_product(url: str) -> str:
    """Extract structured product data with confidence scores from a product URL.
    Use for research calls where a human will review the output."""
    result = extract_with_confidence_routing(url, confidence_threshold=0.8)
    return json.dumps(result, indent=2)


tools = [extract_product, extract_product_for_autofill]

llm = ChatOpenAI(model="gpt-4o")
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a procurement assistant with two extraction modes.\n\n"
            "Choose based on what the human asked:\n\n"
            "- If researching a product or summarizing what's available, call "
            "`extract_product`. Show all fields with confidence scores. Flag any "
            "field below 0.85 as 'verify before relying on'.\n\n"
            "- If filling a purchase order, updating inventory, or writing values "
            "into any system without human review, call `extract_product_for_autofill`. "
            "Default threshold 0.9. Use 0.95 for contract pricing or regulated goods. "
            "If a required field is missing from the autofill response, do NOT invent "
            "it — stop and ask the human to verify manually.\n\n"
            "Never mix modes in a single operation. If the request spans both, "
            "research first, confirm fields, then autofill.",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_openai_functions_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


### Demo: research then autofill

Ask the agent to first research a Moglix product page, then autofill a PO line item. Watch the tool calls: the first uses `extract_product` (all fields with confidence), the second uses `extract_product_for_autofill` with a 0.9 threshold (only high-confidence fields).

In [ ]:
MOGLIX_URL = "https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8"

result = executor.invoke({
    "input": (
        f"Research this product: {MOGLIX_URL}. Show me what's available and "
        f"flag anything low-confidence. Then, assuming the research looks good, "
        f"autofill a PO line item for 3 units at a 0.9 confidence threshold."
    )
})

print(result["output"])


### Why server-side filtering matters

Client-side filtering (checking confidence after the response arrives) still lets the agent *see* low-confidence data. In a long context window, the agent may reference a scrubbed price field anyway because it was visible earlier in the conversation. Server-side filtering removes the temptation entirely — the field never enters the agent's context.

This is the difference between "the agent is trusted to ignore bad data" and "the agent cannot see the bad data." For autonomous writes, the second is the only defensible posture.

## Notes

- Playground: 50 calls/month, no signup required. Starter tier: $99/month for 10K calls with API key. See https://shopgraph.dev/pricing
- Confidence scores range from 0.0 to 1.0, based on extraction method (Schema.org: ~0.93 baseline, LLM: ~0.70 baseline)
- The `_shopgraph.field_freshness` metadata shows whether cached data has decayed — useful for real-time pricing decisions
- Full API documentation: [shopgraph.dev](https://shopgraph.dev)
- Check site extraction status before testing: https://shopgraph.dev/leaderboard
- AgentReady scoring: append `?include_score=true` to score a URL across 6 agent-readiness dimensions
- UCP-compatible output: append `?format=ucp` for Universal Commerce Protocol schema
- Server-side confidence filtering: append `?strict_confidence_threshold=0.85` to scrub low-confidence fields to null
